# Block 2 — candidate generation, INDIA

**Reads:** Block 1 records and labels; the fine-tuned checkpoint
`workspace/block2/artifacts/e5_finetuned_INDIA`.
**Writes:** `workspace/block2/prod/INDIA/candidates/<split>/<ROLE>/cand-*.parquet` (+ `queries-*`,
`pos-*`, `done-*.json`) and the country manifest.

Per-country fine-tuned `multilingual-e5-small` bi-encoder, dense forward search, `K_GEN = 50`
candidates stored per S1. No training here — the checkpoint must already exist.

Run this notebook before `02_block2_candidategen_us.ipynb`, then the merge notebook.
Set `SMOKE = True` for a short end-to-end run into `prod_smoke/` first; check the printed recall
tables, then set `SMOKE = False` and run all. Resumable: committed shards and finished roles are
skipped. GPU.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import gc

import numpy as np
import b2_common as b2
import b2_prod as b2p

In [ ]:
# Config
SMOKE = False        # True first: tiny end-to-end run into prod_smoke/
RUN_TRAIN = True
RUN_TEST = True
b2p.use_smoke_root(SMOKE)
MAX_QUERIES = 3000 if SMOKE else None

TRAIN_COUNTRIES = ["INDIA"]
TEST_COUNTRIES = ["INDIA"]

In [ ]:
# Preflight — GPU/RAM, folder scan, checkpoint verification
infos = b2p.preflight(["INDIA"])
b2p.ensure_prod_root()
for c in set(TRAIN_COUNTRIES + TEST_COUNTRIES):
    b2p.ensure_country_dirs(c)

In [ ]:
# Stage 0 — checkpoint + labels
with b2.step("load fine-tuned model + labels"):
    CKPT = infos[b2p.checkpoint_key_for(TRAIN_COUNTRIES[0])]
    model = b2p.load_model(CKPT)
    pp, roles_df = b2.load_labels()
    positives = b2p.prepare_positives(pp)
    print(f"{len(positives):,} labeled positive pairs")

## Stage 1 — train roles `CE-FIT → CE-DEV → TREE-FIT → TREE-DEV → CALIBRATION → HOLDOUT`
CE-FIT first: Block 4 training needs it.

In [ ]:
# Stage 1 — train roles (labels attached, positives sidecars written)
for c in TRAIN_COUNTRIES:
    if not RUN_TRAIN or b2p.is_done(c, "TRAIN"):
        print(f"[skip] {c} train stage")
        continue
    ck = infos[b2p.checkpoint_key_for(c)]
    with b2.step(f"{c} train pool: load + embed"):
        pool_tr = b2p.load_pool_sorted(c, "train")
        pool_emb_tr = b2p.pool_embeddings(model, ck, c, "train", pool_tr)
        print(f"{c} train pool: {len(pool_tr):,} rows -> {pool_emb_tr.shape}")
    for role in b2p.TRAIN_ROLE_ORDER:
        with b2.step(f"{c} train {role}"):
            b2p.run_role(model, ck, c, "train", role, pool_tr, pool_emb_tr, positives,
                         max_queries=MAX_QUERIES)
    b2p.mark_done(c, "TRAIN", {"checkpoint": ck["fingerprint"], "smoke": SMOKE})
    del pool_tr, pool_emb_tr
    gc.collect()

In [ ]:
# Pair completeness per role, from the positives sidecars
pp_, roles_df_ = b2.load_labels()
bucket_map = dict(zip(b2._get_s1_id_series(roles_df_), roles_df_["match_bucket"].astype(str)))
for c in TRAIN_COUNTRIES:
    shards = [s for s in b2p.collect_shards(c) if s["split"] == "train"]
    pos = b2p.load_pos_sidecars(shards)
    for role in b2p.TRAIN_ROLE_ORDER:
        sub = pos[pos["role"] == role]
        if len(sub):
            b2p.print_pc(b2p.pc_from_positives(sub, bucket_map=bucket_map), f"{c} {role}")
print("\nCE-FIT recall is optimistic: the bi-encoder was fine-tuned on 90% of CE-FIT. "
      "TREE-FIT / TREE-DEV / CALIBRATION / HOLDOUT are the honest estimate.")

## Stage 2 — test

In [ ]:
# Stage 2 — test (no labels; the `label` column is null)
for c in TEST_COUNTRIES:
    if not RUN_TEST or b2p.is_done(c, "TEST"):
        print(f"[skip] {c} test stage")
        continue
    ck = infos[b2p.checkpoint_key_for(c)]
    with b2.step(f"{c} test pool: load + embed"):
        pool_te = b2p.load_pool_sorted(c, "test")
        pool_emb_te = (b2p.pool_embeddings(model, ck, c, "test", pool_te) if len(pool_te)
                       else np.zeros((0, 384), dtype=np.float16))
        print(f"{c} test pool: {len(pool_te):,} rows")
    with b2.step(f"{c} test TEST"):
        b2p.run_role(model, ck, c, "test", b2p.TEST_ROLE, pool_te, pool_emb_te, None,
                     max_queries=MAX_QUERIES)
    b2p.mark_done(c, "TEST", {"checkpoint": ck["fingerprint"], "smoke": SMOKE})
    del pool_te, pool_emb_te
    gc.collect()

In [ ]:
# Summary + distribution-shift sanity: top-1 retrieval score percentiles per split
for c in sorted(set(TRAIN_COUNTRIES + TEST_COUNTRIES)):
    b2p.write_country_manifest(c)
    shards = b2p.collect_shards(c)
    for split in ("train", "test"):
        ss = [s for s in shards if s["split"] == split]
        if not ss:
            continue
        rows = sum(s["n_rows"] for s in ss)
        nq = sum(s["n_queries"] for s in ss)
        med = [s["top1_score_pcts"][2] for s in ss if s.get("top1_score_pcts")]
        print(f"{c:8s} {split:5s}: {nq:>10,} queries  {rows:>12,} candidate rows  "
              f"median top-1 score ~ {np.median(med) if med else float('nan'):.4f}")
print("\nINDIA done. Next: 02_block2_candidategen_us.ipynb")